# Machine Sensor Anomaly Detection

## 1. Project Overview

This project applies unsupervised learning and statistical
methods to detect anomalous behavior in machine sensor data.

The approaches include:

- Exploratory data analysis
- Univariate anomaly detection
- Isolation Forest
- Local Outlier Factor (LOF)
- PCA reconstruction error
- Collective/sequential anomaly detection
- Evaluation and comparison
- Ensemble anomaly detection

In [73]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score

Load the dataset

In [2]:
df = pd.read_csv(
    "../data/machine_sensors.csv",
    parse_dates=["timestamp"]
)
FEATURES = [
    "temperature_c",
    "pressure_kpa",
    "vibration_mm_s",
    "rotation_rpm",
    "power_kw"
]

Explore

In [3]:
df.head()

,timestamp,temperature_c,pressure_kpa,vibration_mm_s,rotation_rpm,power_kw
0,2026-06-01 00:00:00,70.449712,113.504429,2.942719,1679.740512,17.185800
1,2026-06-01 00:01:00,70.378174,116.361526,3.125069,1695.659489,16.844006
2,2026-06-01 00:02:00,71.449114,114.820336,2.950984,1730.242205,17.752159
3,2026-06-01 00:03:00,70.242842,115.215615,3.073366,1739.448509,17.712239
4,2026-06-01 00:04:00,69.036950,114.781110,2.943914,1668.264125,16.330611


In [4]:
df.shape

(8000, 6)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   timestamp       8000 non-null   datetime64[ns]
 1   temperature_c   8000 non-null   float64       
 2   pressure_kpa    8000 non-null   float64       
 3   vibration_mm_s  8000 non-null   float64       
 4   rotation_rpm    8000 non-null   float64       
 5   power_kw        8000 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 375.1 KB


In [6]:
df.describe()

,timestamp,temperature_c,pressure_kpa,vibration_mm_s,rotation_rpm,power_kw
count,8000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,2026-06-03 18:39:29.999999744,70.500803,115.099077,3.095662,1700.711267,17.539689
min,2026-06-01 00:00:00,64.212451,78.429036,2.411117,1624.999922,15.425804
25%,2026-06-02 09:19:45,68.295580,113.988404,2.902147,1685.924604,17.025360
50%,2026-06-03 18:39:30,70.499197,115.159362,3.020649,1701.037213,17.558936
75%,2026-06-05 03:59:15,72.332522,116.275290,3.142708,1715.882225,18.045175
max,2026-06-06 13:19:00,86.372438,120.153655,9.559492,1783.220133,23.409002
std,NaN,2.859595,1.853140,0.462901,21.036400,0.689272


Exploratory Data Analysis

In [7]:
fig, axes = plt.subplots(
    len(FEATURES),
    1,
    figsize=(14, 12),
    sharex=True
)

for ax, col in zip(axes, FEATURES):
    ax.plot(
        df["timestamp"],
        df[col],
        linewidth=0.6
    )
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)

plt.xlabel("Time")
plt.tight_layout()

plt.savefig(
    "../results/exploratory_plot.png",
    dpi=120
)

print("Exploratory plot saved.")

Exploratory plot saved.


Univariate anomaly detection

In [8]:
vibration = df["vibration_mm_s"]
print(vibration.head())
print(vibration.mean())
print(vibration.std())

0    2.942719
1    3.125069
2    2.950984
3    3.073366
4    2.943914
Name: vibration_mm_s, dtype: float64
3.0956617385779546
0.4629008977445851


To know whether it is unusual compared with the machine's recent behavior.

In [9]:
rolling_mean = vibration.rolling(window=30, center=True).mean()
print(rolling_mean.head(40))

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
5          NaN
6          NaN
7          NaN
8          NaN
9          NaN
10         NaN
11         NaN
12         NaN
13         NaN
14         NaN
15    2.979869
16    2.975412
17    2.978389
18    2.978492
19    2.978483
20    2.972349
21    2.969552
22    2.959832
23    2.967251
24    2.972493
25    2.972955
26    2.965647
27    2.959970
28    2.960995
29    2.963479
30    2.965990
31    2.970049
32    2.973239
33    2.993506
34    3.009611
35    3.000977
36    3.018660
37    3.018440
38    3.028208
39    3.023048
Name: vibration_mm_s, dtype: float64


In [10]:
rolling_std = vibration.rolling(window=30, center=True).std()
print(rolling_std.head(40))

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
5          NaN
6          NaN
7          NaN
8          NaN
9          NaN
10         NaN
11         NaN
12         NaN
13         NaN
14         NaN
15    0.153539
16    0.156565
17    0.160314
18    0.160296
19    0.160291
20    0.165108
21    0.165635
22    0.171092
23    0.174540
24    0.172271
25    0.172648
26    0.164483
27    0.156383
28    0.156237
29    0.156126
30    0.156214
31    0.157707
32    0.159428
33    0.179142
34    0.183140
35    0.188903
36    0.160744
37    0.160803
38    0.161776
39    0.161084
Name: vibration_mm_s, dtype: float64


In [14]:
z = (vibration - rolling_mean) / rolling_std

print(z.head(40))

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
5          NaN
6          NaN
7          NaN
8          NaN
9          NaN
10         NaN
11         NaN
12         NaN
13         NaN
14         NaN
15   -0.110006
16    0.198446
17   -0.872323
18   -1.091963
19    0.226402
20   -3.009490
21    0.065692
22   -0.394573
23    0.897794
24    1.082816
25    1.109630
26    0.026460
27   -0.710941
28   -0.401582
29    0.646265
30   -1.004934
31    1.549332
32   -0.120184
33    0.444309
34   -1.363568
35   -0.540395
36   -1.811249
37    0.833380
38   -0.322814
39    0.574821
Name: vibration_mm_s, dtype: float64


Look at our z-scores

In [15]:
print(z.abs().sort_values(ascending=False).head(10))

1197    5.257911
2674    5.253152
3208    5.244399
7050    5.239357
3614    5.233014
2299    5.230500
5129    5.218240
6743    5.199962
1100    5.188183
5192    5.165610
Name: vibration_mm_s, dtype: float64


Choose anomaly threshold

In [17]:
univariate_flags = (z.abs() > 4).astype(int)
print(univariate_flags)

0       0
1       0
2       0
3       0
4       0
       ..
7995    0
7996    0
7997    0
7998    0
7999    0
Name: vibration_mm_s, Length: 8000, dtype: int64


In [18]:
z.abs()

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
7995   NaN
7996   NaN
7997   NaN
7998   NaN
7999   NaN
Name: vibration_mm_s, Length: 8000, dtype: float64

Count the anomalies

In [20]:
print("Univariate anomalies:", univariate_flags.sum())

Univariate anomalies: 10


In [21]:
print("Anomaly percentage:",
      univariate_flags.mean() * 100, "%")

Anomaly percentage: 0.125 %


In [22]:
anomaly_indices = df.index[univariate_flags == 1]

print(anomaly_indices)

Index([1100, 1197, 2299, 2674, 3208, 3614, 5129, 5192, 6743, 7050], dtype='int64')


In [23]:
print(
    df.loc[
        univariate_flags == 1,
        ["timestamp", "vibration_mm_s"]
    ]
)

               timestamp  vibration_mm_s
1100 2026-06-01 18:20:00        6.598718
1197 2026-06-01 19:57:00        8.350911
2299 2026-06-02 14:19:00        7.998459
2674 2026-06-02 20:34:00        9.054844
3208 2026-06-03 05:28:00        8.104588
3614 2026-06-03 12:14:00        8.363975
5129 2026-06-04 13:29:00        9.075286
5192 2026-06-04 14:32:00        9.559492
6743 2026-06-05 16:23:00        7.337642
7050 2026-06-05 21:30:00        7.148475


Step 2 — Isolation Forest

In [24]:
X = df[FEATURES]

print(X.head())

   temperature_c  pressure_kpa  vibration_mm_s  rotation_rpm   power_kw
0      70.449712    113.504429        2.942719   1679.740512  17.185800
1      70.378174    116.361526        3.125069   1695.659489  16.844006
2      71.449114    114.820336        2.950984   1730.242205  17.752159
3      70.242842    115.215615        3.073366   1739.448509  17.712239
4      69.036950    114.781110        2.943914   1668.264125  16.330611


Scale Data

In [25]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled[:5])

[[-0.01786761 -0.86056538 -0.33042038 -0.99694177 -0.51345603]
 [-0.04288604  0.68129127  0.06353227 -0.24015962 -1.00936444]
 [ 0.33164492 -0.15042518 -0.31256502  1.40388965  0.30827211]
 [-0.09021453  0.06289055 -0.04816881  1.84155389  0.25035249]
 [-0.51194123 -0.17159357 -0.32783983 -1.54252493 -1.7542481 ]]


Isolation Forest

In [60]:
iso = IsolationForest(
    contamination=0.03,
    random_state=42
)

fit and predict

In [61]:
iso_predictions = iso.fit_predict(X_scaled)
iso_predictions

array([1, 1, 1, ..., 1, 1, 1], shape=(8000,))

In [62]:
iso_flags = (iso_predictions == -1).astype(int)
iso_flags

array([0, 0, 0, ..., 0, 0, 0], shape=(8000,))

In [63]:
print(iso_flags[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [64]:
print("Isolation Forest anomalies:", iso_flags.sum())

Isolation Forest anomalies: 240


Compare the two algorithms

In [65]:
print("Univariate anomalies:", univariate_flags.sum())
print("Isolation Forest anomalies:", iso_flags.sum())

Univariate anomalies: 10
Isolation Forest anomalies: 240


In [66]:
agreement = pd.DataFrame({
    "zscore": univariate_flags,
    "isolation_forest": iso_flags
})

print(pd.crosstab(
    agreement["zscore"],
    agreement["isolation_forest"]
))

isolation_forest     0    1
zscore                     
0                 7758  232
1                    2    8


Create the LOF model

In [67]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.03
)

Fit and predict
 1  → normal
-1  → anomaly

In [68]:
lof_predictions = lof.fit_predict(X_scaled)
lof_predictions 

array([1, 1, 1, ..., 1, 1, 1], shape=(8000,))

In [69]:
lof_flags = (lof_predictions == -1).astype(int)
print("LOF anomalies:", lof_flags.sum())

LOF anomalies: 240


Compare all three methods

In [70]:
print("Univariate Z-score:", univariate_flags.sum())
print("Isolation Forest:", iso_flags.sum())
print("LOF:", lof_flags.sum())

Univariate Z-score: 10
Isolation Forest: 240
LOF: 240


check they flag the same observation

In [71]:
comparison = pd.DataFrame({
    "zscore": univariate_flags,
    "isolation_forest": iso_flags,
    "lof": lof_flags
})

print(pd.crosstab(
    comparison["isolation_forest"],
    comparison["lof"]
))

lof                  0    1
isolation_forest           
0                 7544  216
1                  216   24


check howmany all three method agree

In [72]:
all_three = (
    (univariate_flags == 1) &
    (iso_flags == 1) &
    (lof_flags == 1)
)

print("All three agree:", all_three.sum())

All three agree: 8


PCA reconstruction erro

In [74]:
pca = PCA(n_components=2)

Project the data into the 2D PCA space

In [77]:
X_pca = pca.fit_transform(X_scaled)
print(X_pca.shape)
print(X_pca[:5])

(8000, 2)
[[-1.16086202  0.45109664]
 [-0.32823567  0.18506518]
 [ 0.81133569 -0.61846007]
 [ 0.91709075 -0.71386843]
 [-2.00290311  0.61135264]]


See how much information the 2 components captured

In [78]:
print(pca.explained_variance_ratio_)

[0.5267635  0.21917496]


In [79]:
print(
    "Total explained variance:",
    pca.explained_variance_ratio_.sum()
)

Total explained variance: 0.7459384597665861


Reconstruct the original data

In [81]:
X_reconstructed = pca.inverse_transform(X_pca)
print(X_reconstructed.shape)

(8000, 5)


Calculate reconstruction error

In [83]:
reconstruction_error = np.mean(
    (X_scaled - X_reconstructed) ** 2,
    axis=1
)
print(reconstruction_error[:10])

[0.11130216 0.28090776 0.25109847 0.42354953 0.29408074 0.01736175
 0.04216102 0.45671111 0.020912   0.22584848]


Choose an anomaly threshold
How large does the reconstruction error need to be before we call it an anomaly?

In [90]:
pca_threshold = np.percentile(
    reconstruction_error,
    97
)
print("PCA threshold:", pca_threshold)

PCA threshold: 0.6722635873199093


create our flags

In [91]:
pca_flags = (
    reconstruction_error > pca_threshold
).astype(int)
print("PCA anomalies:", pca_flags.sum())

PCA anomalies: 240


In [92]:
top_pca = pd.DataFrame({
    "timestamp": df["timestamp"],
    "temperature_c": df["temperature_c"],
    "pressure_kpa": df["pressure_kpa"],
    "vibration_mm_s": df["vibration_mm_s"],
    "rotation_rpm": df["rotation_rpm"],
    "power_kw": df["power_kw"],
    "reconstruction_error": reconstruction_error
})

print(
    top_pca.sort_values(
        "reconstruction_error",
        ascending=False
    ).head(10)
)

               timestamp  temperature_c  pressure_kpa  vibration_mm_s  \
3631 2026-06-03 12:31:00      70.024686     78.429036        2.751227   
7267 2026-06-06 01:07:00      71.568908     81.243901        3.105326   
2014 2026-06-02 09:34:00      72.087509     83.045860        3.054580   
4462 2026-06-04 02:22:00      72.630617     84.776830        2.743639   
2688 2026-06-02 20:48:00      67.576051     86.290209        3.039390   
331  2026-06-01 05:31:00      75.154991     88.986134        2.878569   
4702 2026-06-04 06:22:00      71.829098     93.807382        3.334481   
436  2026-06-01 07:16:00      73.115592     94.305432        3.012293   
2674 2026-06-02 20:34:00      67.308611    116.119322        9.054844   
2016 2026-06-02 09:36:00      71.898287    116.056935        2.947365   

      rotation_rpm   power_kw  reconstruction_error  
3631   1711.943968  16.990452             57.773882  
7267   1704.647778  17.729722             50.431146  
2014   1716.296734  18.135236     

Sequential / collective detection

Create rolling features

In [93]:
window = 30

rolling_features = pd.DataFrame(index=df.index)

for col in FEATURES:
    rolling_features[f"{col}_mean"] = (
        df[col].rolling(window=window).mean()
    )
    
    rolling_features[f"{col}_std"] = (
        df[col].rolling(window=window).std()
    )

print(rolling_features.head())

   temperature_c_mean  temperature_c_std  pressure_kpa_mean  pressure_kpa_std  \
0                 NaN                NaN                NaN               NaN   
1                 NaN                NaN                NaN               NaN   
2                 NaN                NaN                NaN               NaN   
3                 NaN                NaN                NaN               NaN   
4                 NaN                NaN                NaN               NaN   

   vibration_mm_s_mean  vibration_mm_s_std  rotation_rpm_mean  \
0                  NaN                 NaN                NaN   
1                  NaN                 NaN                NaN   
2                  NaN                 NaN                NaN   
3                  NaN                 NaN                NaN   
4                  NaN                 NaN                NaN   

   rotation_rpm_std  power_kw_mean  power_kw_std  
0               NaN            NaN           NaN  
1               NaN 

In [95]:
rolling_features = rolling_features.dropna()
print(rolling_features.shape)

(7971, 10)


In [96]:
rolling_scaler = StandardScaler()

rolling_scaled = rolling_scaler.fit_transform(
    rolling_features
)

Apply Isolation Forest

In [97]:
collective_model = IsolationForest(
    contamination=0.03,
    random_state=42
)

collective_predictions = collective_model.fit_predict(
    rolling_scaled
)

In [98]:
collective_window_flags = (
    collective_predictions == -1
).astype(int)

In [99]:
print(
    "Collective window anomalies:",
    collective_window_flags.sum()
)

Collective window anomalies: 240


we need to put the results back onto the original 8,000-row index.

In [101]:
collective_flags = np.zeros(len(df), dtype=int)

collective_flags[
    rolling_features.index
] = collective_window_flags

print("Total collective anomalies:", collective_flags.sum())

Total collective anomalies: 240


In [102]:
print(len(collective_flags))

8000


Which method actually detects the labeled anomalies correctly?
Precision — when the model says anomaly, how often is it correct?
Recall — how many of the real anomalies did it find?
F1-score — balance between precision and recall.

In [104]:
import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if "label" in file.lower() and file.endswith(".csv"):
            print(os.path.join(root, file))

In [105]:
import os

print(os.listdir("."))
print(os.listdir("../"))

['anomaly_detection.ipynb']
['.gitignore', 'anomalyDetection.ipynb', 'data', 'exercise.pdf', 'exploratory_plot.png', 'models', 'notebooks', 'README.md', 'requirement.txt', 'results', 'src', 'starter.py', 'UI']


In [106]:
os.listdir("../data")

['machine_sensors.csv']

In [107]:
# ---------------------------------------------------------------------------
# 5. Compare detection methods
# ---------------------------------------------------------------------------

results = df.copy()

results["zscore"] = univariate_flags
results["isolation_forest"] = iso_flags
results["lof"] = lof_flags
results["pca"] = pca_flags
results["collective"] = collective_flags

In [108]:
print("Univariate Z-score:", results["zscore"].sum())
print("Isolation Forest:", results["isolation_forest"].sum())
print("LOF:", results["lof"].sum())
print("PCA:", results["pca"].sum())
print("Collective:", results["collective"].sum())

Univariate Z-score: 10
Isolation Forest: 240
LOF: 240
PCA: 240
Collective: 240


In [109]:
results["anomaly_votes"] = (
    results["zscore"]
    + results["isolation_forest"]
    + results["lof"]
    + results["pca"]
    + results["collective"]
)

print(
    results["anomaly_votes"]
    .value_counts()
    .sort_index()
)

anomaly_votes
0    7264
1     537
2     173
3      18
4       7
5       1
Name: count, dtype: int64


5 → very strong agreement
4 → strong agreement
3 → moderate agreement
1 → weak agreement
0 → no detector flagged it

In [112]:
readable = strong_anomalies[
    [
        "timestamp",
        "vibration_mm_s",
        "pressure_kpa",
        "power_kw",
        "zscore",
        "isolation_forest",
        "lof",
        "pca",
        "collective",
        "anomaly_votes"
    ]
].sort_values("anomaly_votes", ascending=False)

readable

,timestamp,vibration_mm_s,pressure_kpa,power_kw,zscore,isolation_forest,lof,pca,collective,anomaly_votes
5192,2026-06-04 14:32:00,9.559492,113.593957,17.199164,1,1,1,1,1,5
1197,2026-06-01 19:57:00,8.350911,113.772807,16.751831,1,1,1,1,0,4
2299,2026-06-02 14:19:00,7.998459,115.092995,16.643131,1,1,1,1,0,4
2674,2026-06-02 20:34:00,9.054844,116.119322,16.789881,1,1,1,1,0,4
3208,2026-06-03 05:28:00,8.104588,117.506624,18.369609,1,1,1,1,0,4
3614,2026-06-03 12:14:00,8.363975,113.775609,17.421914,1,1,1,1,0,4
6743,2026-06-05 16:23:00,7.337642,113.446917,16.705434,1,1,1,1,0,4
5129,2026-06-04 13:29:00,9.075286,116.177675,17.167522,1,1,1,1,0,4
331,2026-06-01 05:31:00,2.878569,88.986134,17.684191,0,1,1,1,0,3
5161,2026-06-04 14:01:00,4.546082,110.503656,17.733351,0,1,1,1,0,3


In [113]:
# Display individual detector decisions

readable = strong_anomalies[
    [
        "timestamp",
        "temperature_c",
        "pressure_kpa",
        "vibration_mm_s",
        "rotation_rpm",
        "power_kw",
        "zscore",
        "isolation_forest",
        "lof",
        "pca",
        "collective",
        "anomaly_votes"
    ]
].copy()

# Make the column names easier to read
readable = readable.rename(columns={
    "temperature_c": "Temperature",
    "pressure_kpa": "Pressure",
    "vibration_mm_s": "Vibration",
    "rotation_rpm": "RPM",
    "power_kw": "Power",
    "zscore": "Z-Score",
    "isolation_forest": "IF",
    "lof": "LOF",
    "pca": "PCA",
    "collective": "Collective",
    "anomaly_votes": "Votes"
})

# Sort strongest agreement first
readable = readable.sort_values(
    "Votes",
    ascending=False
)

readable

,timestamp,Temperature,Pressure,Vibration,RPM,Power,Z-Score,IF,LOF,PCA,Collective,Votes
5192,2026-06-04 14:32:00,77.824722,113.593957,9.559492,1693.715492,17.199164,1,1,1,1,1,5
1197,2026-06-01 19:57:00,68.138543,113.772807,8.350911,1693.648469,16.751831,1,1,1,1,0,4
2299,2026-06-02 14:19:00,67.913510,115.092995,7.998459,1686.598807,16.643131,1,1,1,1,0,4
2674,2026-06-02 20:34:00,67.308611,116.119322,9.054844,1667.833599,16.789881,1,1,1,1,0,4
3208,2026-06-03 05:28:00,74.194572,117.506624,8.104588,1716.214363,18.369609,1,1,1,1,0,4
3614,2026-06-03 12:14:00,69.046528,113.775609,8.363975,1690.994443,17.421914,1,1,1,1,0,4
6743,2026-06-05 16:23:00,67.223600,113.446917,7.337642,1678.361482,16.705434,1,1,1,1,0,4
5129,2026-06-04 13:29:00,73.677728,116.177675,9.075286,1705.538789,17.167522,1,1,1,1,0,4
331,2026-06-01 05:31:00,75.154991,88.986134,2.878569,1736.910427,17.684191,0,1,1,1,0,3
5161,2026-06-04 14:01:00,75.226312,110.503656,4.546082,1668.729255,17.733351,0,1,1,1,0,3


investigate the anomaly types

Step 1 — Separate strong candidates

In [114]:
# Strong agreement between detectors

strong_3 = results[results["anomaly_votes"] >= 3]
strong_4 = results[results["anomaly_votes"] >= 4]
strong_5 = results[results["anomaly_votes"] == 5]

print("3+ detectors:", len(strong_3))
print("4+ detectors:", len(strong_4))
print("5 detectors:", len(strong_5))

3+ detectors: 26
4+ detectors: 8
5 detectors: 1


Step 2 — Investigate the vibration spikes
point anomalies caused by sudden vibration spikes.

In [115]:
vibration_anomalies = df[
    df["vibration_mm_s"] > 6
].copy()

vibration_anomalies[
    [
        "timestamp",
        "temperature_c",
        "pressure_kpa",
        "vibration_mm_s",
        "rotation_rpm",
        "power_kw"
    ]
].sort_values(
    "vibration_mm_s",
    ascending=False
).head(20)

,timestamp,temperature_c,pressure_kpa,vibration_mm_s,rotation_rpm,power_kw
5192,2026-06-04 14:32:00,77.824722,113.593957,9.559492,1693.715492,17.199164
5129,2026-06-04 13:29:00,73.677728,116.177675,9.075286,1705.538789,17.167522
2674,2026-06-02 20:34:00,67.308611,116.119322,9.054844,1667.833599,16.789881
3614,2026-06-03 12:14:00,69.046528,113.775609,8.363975,1690.994443,17.421914
1197,2026-06-01 19:57:00,68.138543,113.772807,8.350911,1693.648469,16.751831
3208,2026-06-03 05:28:00,74.194572,117.506624,8.104588,1716.214363,18.369609
2299,2026-06-02 14:19:00,67.913510,115.092995,7.998459,1686.598807,16.643131
6743,2026-06-05 16:23:00,67.223600,113.446917,7.337642,1678.361482,16.705434
7050,2026-06-05 21:30:00,68.427178,115.175468,7.148475,1699.492408,17.667355
1100,2026-06-01 18:20:00,68.853658,114.066161,6.598718,1687.126221,17.129846


Step 3 — Investigate pressure drops

In [116]:
pressure_anomalies = df[
    df["pressure_kpa"] < 100
].copy()

pressure_anomalies[
    [
        "timestamp",
        "temperature_c",
        "pressure_kpa",
        "vibration_mm_s",
        "rotation_rpm",
        "power_kw"
    ]
].sort_values(
    "pressure_kpa"
)

,timestamp,temperature_c,pressure_kpa,vibration_mm_s,rotation_rpm,power_kw
3631,2026-06-03 12:31:00,70.024686,78.429036,2.751227,1711.943968,16.990452
7267,2026-06-06 01:07:00,71.568908,81.243901,3.105326,1704.647778,17.729722
2014,2026-06-02 09:34:00,72.087509,83.045860,3.054580,1716.296734,18.135236
4462,2026-06-04 02:22:00,72.630617,84.776830,2.743639,1704.605551,17.810574
2688,2026-06-02 20:48:00,67.576051,86.290209,3.039390,1716.594306,17.508915
331,2026-06-01 05:31:00,75.154991,88.986134,2.878569,1736.910427,17.684191
4702,2026-06-04 06:22:00,71.829098,93.807382,3.334481,1721.842321,17.630773
436,2026-06-01 07:16:00,73.115592,94.305432,3.012293,1704.884877,18.144908


Step 4 — Investigate the contextual anomaly

In [117]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    df["rotation_rpm"],
    df["power_kw"],
    s=8
)

plt.xlabel("Rotation (RPM)")
plt.ylabel("Power (kW)")
plt.title("Power vs Rotation Speed")

plt.tight_layout()
plt.savefig("power_vs_rpm.png", dpi=100)

Step 5 — Find the suspicious power observations

In [118]:
df[
    [
        "timestamp",
        "rotation_rpm",
        "power_kw",
        "temperature_c",
        "pressure_kpa",
        "vibration_mm_s"
    ]
].sort_values(
    "power_kw",
    ascending=False
).head(20)

,timestamp,rotation_rpm,power_kw,temperature_c,pressure_kpa,vibration_mm_s
2016,2026-06-02 09:36:00,1702.341759,23.409002,71.898287,116.056935,2.947365
3145,2026-06-03 04:25:00,1727.430230,23.308303,72.363242,118.216135,2.904285
2878,2026-06-02 23:58:00,1688.067575,22.005560,69.820458,112.863111,3.018701
5455,2026-06-04 18:55:00,1678.924265,21.687402,67.277340,112.909922,2.869981
2510,2026-06-02 17:50:00,1664.644295,21.634610,66.205025,114.410199,2.872919
7131,2026-06-05 22:51:00,1688.788876,21.460727,70.051454,115.428467,3.114782
4546,2026-06-04 03:46:00,1697.259609,21.266129,70.226632,116.141095,2.969638
2000,2026-06-02 09:20:00,1721.218285,21.221763,71.798214,116.592114,3.038713
5351,2026-06-04 17:11:00,1668.962832,20.887108,66.657263,115.636551,3.186254
5615,2026-06-04 21:35:00,1688.711605,20.782986,68.642913,115.612335,2.849465


Step 6 — Investigate the collective anomaly

In [119]:
window = df[
    (df["timestamp"] >= "2026-06-06 03:00:00") &
    (df["timestamp"] <= "2026-06-06 05:30:00")
]

plt.figure(figsize=(14, 6))

plt.plot(
    window["timestamp"],
    window["temperature_c"],
    label="Temperature"
)

plt.plot(
    window["timestamp"],
    window["vibration_mm_s"],
    label="Vibration"
)

plt.xlabel("Time")
plt.ylabel("Sensor Value")
plt.title("Potential Collective Anomaly")
plt.legend()

plt.tight_layout()
plt.savefig("collective_anomaly.png", dpi=100)